# Module 3 — Class 1: Data Quality and Cleaning

**Lecture reference:** `MODULE_3_OLIST_REWRITE.md` § Class 3-1

**Today:** load the raw Olist orders + customers tables. Find every kind of dirt. Fix it. Ship `orders_step1.parquet` so Class 3-2 can pick up where you stopped.

## Why this matters today

The classifier you'll train in Module 4 (predicting late deliveries) is only as good as today's cleaning. Garbage in → garbage out. The dataset you'll use for the next 6 weeks of teaching lands on your laptop today.

## Setup

1. Download Olist from Kaggle: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
2. Unzip into a folder named `olist_data/` next to this notebook.
3. Run the next cell to verify all 9 CSVs are present.

In [ ]:
import os
expected = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'product_category_name_translation.csv',
]
for f in expected:
    print(('✓' if os.path.exists(f'olist_data/{f}') else '✗'), f)

## Section 1 — Load the two starter tables

In [ ]:
import pandas as pd
import numpy as np

orders = pd.read_csv('olist_data/olist_orders_dataset.csv')
customers = pd.read_csv('olist_data/olist_customers_dataset.csv')

print('orders   :', orders.shape)        # expect (99441, 8)
print('customers:', customers.shape)     # expect (99441, 5)
orders.head()

### The 5 inspection commands from Module 2

In [ ]:
orders.head()

In [ ]:
orders.shape

In [ ]:
orders.dtypes

In [ ]:
orders.isna().sum()

In [ ]:
orders.describe(include='all')

**📝 Write down what you noticed in this markdown cell:**

- ___ rows, ___ columns
- The `order_*_timestamp` columns are stored as ___ but should be ___
- ___ rows have NaN in `order_delivered_customer_date`
- `order_status` has these unique values: ___

## Section 2 — Parse the dates (always with `errors='coerce'`)

In [ ]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

print(orders.dtypes[date_cols])

**Why `errors='coerce'`?**  
Real-world dates have typos. `coerce` turns unparseable values into `NaT` instead of crashing. Always use it on customer-supplied dates.

## Section 3 — Investigate missing delivery dates BEFORE deciding what to do

In [ ]:
# What kind of orders have no delivery date?
missing_delivery = orders[orders['order_delivered_customer_date'].isna()]
print(missing_delivery['order_status'].value_counts())
print('\nFraction missing:',
      orders['order_delivered_customer_date'].isna().mean().round(4))

**📝 Decision (write yours below):**

Rows with missing `order_delivered_customer_date` are mostly `___`, `___`, and `___`. They represent orders that never reached the customer.  
Decision: **keep them in the analytical table** (we'll need them in M5) but **filter them out of the M4 modelling set** because we can't compute `is_late` without a delivery date.

## Section 4 — Duplicates check (habit)

In [ ]:
print('Full-row duplicates:', orders.duplicated().sum())
print('Unique order_ids equals total rows:',
      orders['order_id'].nunique() == len(orders))

## Section 5 — Filter to the statuses we'll model

In [ ]:
# Statuses we keep for the M4 modelling task (will compute is_late on these)
model_statuses = {'delivered', 'invoiced', 'shipped', 'approved'}
orders_model = orders[orders['order_status'].isin(model_statuses)].copy()

# Anything with no delivery date in this subset → drop (can't model)
before = len(orders_model)
orders_model = orders_model.dropna(subset=['order_delivered_customer_date'])
after = len(orders_model)
print(f'Rows kept for modelling: {after} (dropped {before - after} with no delivery date)')

# Keep the dropped ones in a separate file (M5 segmentation may use them)
orders_unmodellable = orders[~orders.index.isin(orders_model.index)].copy()
print(f'Rows for the side file: {len(orders_unmodellable)}')

## Section 6 — Save the cleaned starter file (Stage 1 of the M3 pipeline)

In [ ]:
orders_model.to_parquet('orders_step1.parquet', index=False)
orders_unmodellable.to_parquet('orders_undelivered.parquet', index=False)
print('Saved.')
print('orders_step1.parquet shape:', orders_model.shape)
print('orders_undelivered.parquet shape:', orders_unmodellable.shape)

## ✅ Quick Check

1. What does `errors='coerce'` do that's different from default?  
2. Why is missing `order_delivered_customer_date` *signal*, not just *noise*?  
3. After today's filter you should have roughly how many rows in `orders_step1.parquet`?

## 📝 Homework (Bronze)

1. Run every cell above end-to-end. No errors.  
2. Fill in the 4 markdown cells where I asked for your observations.  
3. Push your notebook + `orders_step1.parquet` to `module-3/class_1/submissions/<YourName>/`.


## 🔥 Homework (Gold) — same dataset, harder ask

1. Also clean the `customers` table (lowercase state codes, drop duplicate customer_unique_id rows keeping the most recent).  
2. Merge `orders` + `customers` on `customer_id`. Save as `orders_customers_step1.parquet`.  
3. Half-page markdown rationale: every cleaning decision you made and why.